# Week 08 — Home exercise 3: One year of Apple

**Solution proposal.**

Dates, `resample`, and two ways of measuring a fall that disagree about which day was worst.

In [1]:
import pandas as pd

apple = pd.read_csv("../data/AAPL.csv")
apple["Date"] = pd.to_datetime(apple["Date"])

apple_dated = apple.set_index("Date").sort_index()

print(apple_dated.shape)
print(apple_dated.index.min().date(), "to", apple_dated.index.max().date())

(252, 6)
2020-01-02 to 2020-12-30


252 trading days. The file stops on 30 December, not 31 — worth noticing before writing anything that
assumes a complete year.

## 1. Mean closing price by quarter

In [2]:
apple_dated["Close"].resample("QE").mean().round(2)

Date
2020-03-31     73.54
2020-06-30     77.49
2020-09-30    109.12
2020-12-31    120.08
Freq: QE-DEC, Name: Close, dtype: float64

**Q4 was the strongest**, at 120.08. The shape of the year is the whole pandemic story in four
numbers: a first quarter dragged down by the March crash, then three quarters of recovery that end
well above where the year started.

## 2. Total volume by month

In [3]:
volume_by_month = apple_dated["Volume"].resample("ME").sum()

volume_by_month.sort_values(ascending=False).head(3)

Date
2020-03-31    6280072400
2020-08-31    4070623100
2020-09-30    3885767100
Name: Volume, dtype: int64

**March**, by a wide margin — 6.3 billion shares against 4.1 billion in the next busiest month.

That is the crash. Volume measures how many shares changed hands, so it spikes when people are
reacting rather than holding: March 2020 is when the market decided the pandemic was real. Note that
the busiest month and the worst-performing quarter are the same period, which is not a coincidence.

## 3. Weekly closing prices

In [4]:
weekly = apple_dated["Close"].resample("W").last()

print("rows:", len(weekly))
weekly.head(3).round(2)

rows: 53


Date
2020-01-05    74.36
2020-01-12    77.58
2020-01-19    79.68
Freq: W-SUN, Name: Close, dtype: float64

**53 rows, not 52.**

`resample` builds its groups from the calendar, and 2020's calendar does not divide into 52 whole
weeks starting on 1 January. Weeks here are labeled by the Sunday they end on: the first runs to
5 January and holds only two trading days, because the year began on a Wednesday, and the last is
labeled **3 January 2021** and holds the three trading days left at the end of December. Two partial
weeks bracketing the full ones is 53 periods, not 52.

This is `resample` doing the thing it exists for. A `groupby` on a week number would have given
whatever the data happened to contain; `resample` gives what the calendar contains, and then tells you
how much of each period was filled.

## 4. The worst day, measured two ways

In [5]:
apple_dated["change"] = apple_dated["Close"].diff()
apple_dated["pct_change"] = apple_dated["Close"].pct_change() * 100

print("worst by absolute change:")
print(apple_dated["change"].sort_values().head(3).round(2))
print()
print("worst by percentage change:")
print(apple_dated["pct_change"].sort_values().head(3).round(2))

worst by absolute change:
Date
2020-09-03   -10.52
2020-03-16    -8.94
2020-09-08    -8.14
Name: change, dtype: float64

worst by percentage change:
Date
2020-03-16   -12.86
2020-03-12    -9.88
2020-09-03    -8.01
Name: pct_change, dtype: float64


**By dollars lost: 3 September, down 10.52. By percentage: 16 March, down 12.86%.**

They disagree because the share was worth very different amounts on those two days. On 16 March it
fell from about 69 to 60.55; on 3 September it fell from about 131 to 120.88. A **9-dollar** fall
from 69 is nearly 13%; a **10-dollar** fall from 131 is only 8%. The bigger loss in dollars is the
smaller one in percent.

In [6]:
print(apple_dated.loc["2020-03-16", ["Close", "change", "pct_change"]].round(2))
print()
print(apple_dated.loc["2020-09-03", ["Close", "change", "pct_change"]].round(2))

Close         60.55
change        -8.94
pct_change   -12.86
Name: 2020-03-16 00:00:00, dtype: float64

Close         120.88
change        -10.52
pct_change     -8.01
Name: 2020-09-03 00:00:00, dtype: float64


Which measure is right depends on the question, and neither is a default:

- **Percentage** is right for comparing across time, or across shares with different prices. It is
  what "the market fell 3% today" means, and it is why March wins.
- **Absolute** is right when you want to know what happened to a specific amount of money. If you held
  a thousand shares, September cost you more.

The general point is worth more than the example: a ranking is a claim, and changing how you measure
can change who is at the top without anything being wrong.

## 5. Up days and down days

In [7]:
up = (apple_dated["change"] > 0).sum()
down = (apple_dated["change"] < 0).sum()

first_close = apple_dated["Close"].iloc[0]
last_close = apple_dated["Close"].iloc[-1]

print("days up:  ", up)
print("days down:", down)
print(f"year:      {first_close:.2f} -> {last_close:.2f}  ({(last_close / first_close - 1) * 100:.1f}%)")

days up:   136
days down: 114
year:      75.09 -> 133.72  (78.1%)


**136 up days against 114 down — and that does not explain a 78% year on its own.**

136 to 114 is close to a coin toss: 54% of days were up. A share that rose on 54% of days by the same
amount it fell on the other 46% would end the year up a few percent, not most of the way to double.

The gain came from the **size** of the moves, not their count. Counting the direction of each day
throws away exactly the information that mattered.

> 💡 This is a small instance of a general habit: when a count and a total tell different stories,
the total is usually the one answering your question, and the count is usually the one that was easier
to compute.

### Things worth noticing

- **Set the index and sort it before resampling.** `resample` on an unsorted index does not raise; it
  produces something wrong. `.set_index("Date").sort_index()` as one expression is the habit.
- `.pct_change()` is `.diff()` divided by the previous value, and comes back as a fraction, so the
  `* 100` is yours to remember.
- `.loc["2020-03-16"]` works because the index is dates — you can look a row up by the day it
  happened, which is most of the reason to put dates in the index at all.

### What this notebook does NOT do

- It treats every gap between adjacent rows as one step. `.diff()` between Friday and Monday is a
  three-day change and is reported alongside one-day changes without distinction. For daily prices
  that is the convention; for irregular data it would be a real error.
- One share, one year. Nothing here is evidence about anything except Apple in 2020, and the
  temptation to generalize from a single series that went up 78% is exactly how people lose money.
- It ignores `Adj Close`, which corrects for splits and dividends. Apple split four-for-one in August
  2020 — this file is already adjusted, but on a raw price series that split would appear as a
  catastrophic 75% one-day crash that never happened.